# 🎯 Semantic Re-Ranking 실습

Azure AI Search의 Semantic Re-Ranking 기능을 활용하여 **하이브리드 검색 결과를 의미 기반으로 재정렬**하는 방법을 학습합니다.

## 학습 목표
1. 하이브리드 검색(RRF)의 한계점 이해
2. Semantic Re-Ranking이 해결하는 문제
3. Semantic Configuration 설정 방법
4. 하이브리드 vs 하이브리드+Re-Ranking 결과 비교
5. Captions와 Answers 활용

---
## 1️⃣ 환경 설정

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
)
from azure.search.documents.models import (
    QueryType, 
    QueryCaptionType, 
    QueryAnswerType,
    VectorizedQuery
)
from openai import AzureOpenAI

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")

# Azure OpenAI 설정 (임베딩용)
OPENAI_ENDPOINT = os.getenv("AZURE_OPEN_AI_ENDPOINT")
OPENAI_KEY = os.getenv("AZURE_OPEN_AI_KEY")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

credential = AzureKeyCredential(SEARCH_ADMIN_KEY)

# OpenAI Client 초기화
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_KEY,
    api_version=API_VERSION
)

print(f"✅ Endpoint: {SEARCH_ENDPOINT}")
print(f"✅ Index: {INDEX_NAME}")
print(f"✅ Embedding Model: {EMBEDDING_DEPLOYMENT}")

✅ Endpoint: https://foundryiq-aisearch-260114-changjuahn.search.windows.net
✅ Index: products-index
✅ Embedding Model: text-embedding-3-small


---
## 2️⃣ 하이브리드 검색의 한계와 Re-Ranking의 필요성

### 하이브리드 검색(RRF)의 한계점

하이브리드 검색은 **BM25(키워드)**와 **Vector(벡터)** 검색의 결과를 **RRF(Reciprocal Rank Fusion)**로 결합합니다. 하지만 RRF는 **단순 순위 결합** 알고리즘이기 때문에 몇 가지 한계가 있습니다.

```
❌ 하이브리드 검색(RRF)의 한계:

1. 순위만 결합, 의미는 모름
   - BM25 1위 + Vector 3위 → RRF 점수 계산
   - 실제로 쿼리 의도에 맞는지는 알 수 없음

2. 문서 내용과 쿼리의 "진짜 관련성" 평가 불가
   - "따뜻하고 가벼운 자켓" 검색 시
   - 키워드 매칭: "자켓" 포함된 상품
   - 벡터 유사도: 자켓과 비슷한 임베딩
   - 하지만 리뷰에 "실제로 따뜻해요"가 있는지는 모름

3. 복합적인 쿼리 의도 파악 어려움
   - "아이 생일 선물용으로 좋은 유아용품"
   - RRF는 각 단어/벡터 매칭의 순위만 결합
   - "선물용으로 좋은"이라는 의도를 이해하지 못함
```

### Semantic Re-Ranking이 해결하는 문제

**Semantic Re-Ranking**은 하이브리드 검색 결과를 Microsoft의 **언어 모델**로 다시 평가하여 **쿼리와 문서의 실제 의미적 관련성**을 기준으로 재정렬합니다.

```
검색 흐름:
┌──────────────────┐     ┌──────────────────┐
│ 1. BM25 검색      │     │ 2. Vector 검색    │
│ (키워드 매칭)     │     │ (임베딩 유사도)   │
└────────┬─────────┘     └────────┬─────────┘
         │                         │
         └────────┬────────────────┘
                  ↓
         ┌──────────────────┐
         │ 3. RRF 결합       │ → 상위 50개 결과 (순위 기반 결합)
         └────────┬─────────┘
                  ↓
         ┌──────────────────┐
         │ 4. L2 Re-Rank    │ → 언어 모델로 의미적 관련성 재평가
         └────────┬─────────┘   (쿼리 ↔ 문서 내용 직접 비교)
                  ↓
         ┌──────────────────┐
         │ 5. Captions      │ → 관련 문장 자동 추출
         └──────────────────┘
```

### 비즈니스 예시: "따뜻하고 가벼운 겨울 자켓"

| 단계 | 검색 방식 | 동작 |
|------|----------|------|
| 1 | **BM25** | "따뜻", "가벼운", "겨울", "자켓" 키워드 매칭 |
| 2 | **Vector** | 쿼리 임베딩과 유사한 상품 임베딩 검색 |
| 3 | **RRF** | BM25 순위 + Vector 순위 결합 → 상위 50개 |
| 4 | **Re-Ranking** | 언어 모델이 리뷰 내용 분석: "정말 따뜻해요", "가벼워서 좋아요" → 상위로 재정렬 |

### @search.rerankerScore

- **범위**: 0 ~ 4
- **의미**: 쿼리와 문서의 **실제 의미적 관련성** 점수
- **4에 가까울수록**: 쿼리 의도에 정확히 부합

---
## 3️⃣ 기존 인덱스에 Semantic Configuration 추가

In [9]:
# 인덱스 클라이언트 초기화
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

# 기존 인덱스 가져오기
existing_index = index_client.get_index(INDEX_NAME)

print(f"✅ 기존 인덱스 '{INDEX_NAME}' 로드 완료")
print(f"   - 필드 수: {len(existing_index.fields)}")

✅ 기존 인덱스 'products-index' 로드 완료
   - 필드 수: 8


### Semantic Configuration의 3가지 필드 유형

Semantic Re-Ranking에서는 언어 모델이 참고할 필드를 **고정된 3가지 유형**으로 분류합니다:

| 필드 유형 | 개수 제한 | 가중치 | 역할 |
|----------|----------|--------|------|
| **titleField** | 1개만 지정 | ⭐⭐⭐⭐⭐ (매우 높음) | 문서의 제목/주제를 나타내는 필드 |
| **contentFields** | 최대 여러 개 | ⭐⭐⭐ (중간) | 본문 내용, 상세 설명 등 |
| **keywordFields** | 최대 여러 개 | ⭐⭐ (낮음) | 카테고리, 태그 등 키워드 |

### ⚠️ 왜 필드 유형이 고정되어 있나?

- **언어 모델이 문서 구조를 이해하기 위해**: 제목 vs 본문 vs 키워드의 역할 차이를 모델이 인식
- **자동 가중치 적용**: 제목이 본문보다 중요하다는 것을 모델이 알고 있음
- **최적화된 Re-Ranking**: 필드의 성격에 맞는 언어 이해 전략 적용

### 우리 데이터셋에 적용

```
titleField:      "title"          → 상품명 (가장 중요)
contentFields:   "review", "brand" → 리뷰 내용과 브랜드명 (의미적 맥락)
keywordFields:   "category"        → 카테고리 (분류 정보)
```

💡 **핵심**: 이 3가지 유형은 Azure AI Search의 표준 구조이며, 사용자가 임의로 다른 유형을 만들 수 없습니다.

In [11]:
# Semantic Configuration 정의
semantic_config = SemanticConfiguration(
    name="products-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),  # 제목 필드 (가중치 높음)
        content_fields=[
            SemanticField(field_name="review"),  # 리뷰 내용
            SemanticField(field_name="brand")    # 브랜드명
        ],
        keywords_fields=[
            SemanticField(field_name="category")  # 카테고리 (키워드)
        ]
    )
)

# Semantic Search 설정 추가
semantic_search = SemanticSearch(configurations=[semantic_config])

# 인덱스에 Semantic Search 추가
existing_index.semantic_search = semantic_search

# 인덱스 업데이트
result = index_client.create_or_update_index(existing_index)

print("\n✅ Semantic Configuration 추가 완료!")
print(f"   - Configuration Name: {semantic_config.name}")
print(f"   - Title Field: {semantic_config.prioritized_fields.title_field.field_name}")
print(f"   - Content Fields: {[f.field_name for f in semantic_config.prioritized_fields.content_fields]}")
print(f"   - Keywords Fields: {[f.field_name for f in semantic_config.prioritized_fields.keywords_fields]}")


✅ Semantic Configuration 추가 완료!
   - Configuration Name: products-semantic-config
   - Title Field: title
   - Content Fields: ['review', 'brand']
   - Keywords Fields: ['category']


---
## 4️⃣ 기존 데이터 확인

In [12]:
# 검색 클라이언트 초기화
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

# 기존 데이터 확인
result = search_client.search(search_text="*", top=1, include_total_count=True)
doc_count = result.get_count()

print(f"✅ 기존 인덱스 '{INDEX_NAME}'에 {doc_count}개의 문서가 존재합니다.")
print(f"   → 이전 실습에서 업로드한 데이터를 사용합니다.")

✅ 기존 인덱스 'products-index'에 247개의 문서가 존재합니다.
   → 이전 실습에서 업로드한 데이터를 사용합니다.


---
## 5️⃣ 유틸리티 함수 정의

In [16]:
def get_embedding(text):
    """텍스트를 벡터로 변환"""
    response = openai_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding

def search_comparison(query, top=5):
    """Hybrid vs Hybrid+Semantic 검색 결과 비교"""
    
    print(f"\n{'='*80}")
    print(f" 검색어: '{query}'")
    print(f"{'='*80}")
    
    # 쿼리 임베딩 생성
    query_vector = get_embedding(query)
    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=50, fields="content_vector")
    
    # 1. 하이브리드 검색 (BM25 + Vector + RRF)
    print("\n📌 하이브리드 검색 (BM25 + Vector + RRF)")
    print("-" * 80)
    hybrid_results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        top=top
    )
    
    hybrid_list = []
    for idx, result in enumerate(hybrid_results, 1):
        print(f"{idx}. {result['title'][:50]}...")
        print(f"   브랜드: {result['brand']} | 카테고리: {result['category']}")
        print(f"   가격: {result['normal_price']:,}원 | RRF 점수: {result['@search.score']:.4f}")
        hybrid_list.append(result)
    
    # 2. 하이브리드 + Semantic Re-Ranking
    print("\n📌 하이브리드 + Semantic Re-Ranking")
    print("-" * 80)
    semantic_results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="products-semantic-config",
        query_caption=QueryCaptionType.EXTRACTIVE,
        top=top
    )
    
    semantic_list = []
    for idx, result in enumerate(semantic_results, 1):
        print(f"{idx}. {result['title'][:50]}...")
        print(f"   브랜드: {result['brand']} | 카테고리: {result['category']}")
        print(f"   가격: {result['normal_price']:,}원 | Reranker 점수: {result.get('@search.reranker_score', 0):.4f}")
        
        # Caption 출력 (있는 경우)
        captions = result.get('@search.captions')
        if captions:
            caption_text = captions[0].text if len(captions) > 0 else ""
            if caption_text:
                print(f"   💬 Caption: {caption_text[:100]}...")
        
        semantic_list.append(result)
    
    return hybrid_list, semantic_list

---
## 6️⃣ Semantic Re-Ranking 비교 실습

RRF(순위 결합)만으로는 부족한 상황에서 Re-Ranking이 어떻게 결과를 개선하는지 확인합니다.

### 실습 1: 리뷰 내용 이해가 필요한 검색

**비즈니스 상황**: 고객이 "따뜻하고 가벼운" 제품을 찾을 때, 실제 리뷰에서 그런 평가를 받은 상품이 상위에 와야 함

- **하이브리드(RRF)**: "따뜻", "가벼운" 키워드 매칭 + 벡터 유사도의 순위 결합
- **Re-Ranking**: 리뷰 내용을 분석하여 "정말 따뜻해요", "가벼워서 좋아요" 같은 실제 평가가 있는 상품 우선

In [17]:
# 시나리오 1: 리뷰 내용 이해가 필요한 검색
print("\n" + "="*80)
print(" 시나리오 1: 리뷰 기반 품질 검색")
print(" → RRF는 순위만 결합, Re-Ranking은 리뷰 내용까지 이해")
print("="*80)

hybrid_1, semantic_1 = search_comparison("따뜻하고 가벼운 자켓", top=5)

print("\n💡 분석:")
print("   - 하이브리드(RRF): '따뜻', '가벼운', '자켓' 키워드+벡터 순위 결합")
print("   - Re-Ranking: 리뷰에서 '따뜻해요', '가벼워요' 실제 평가 내용을 이해하여 재정렬")
print("   → 리뷰에 긍정적 평가가 있는 상품이 상위로 이동")


 시나리오 1: 리뷰 기반 품질 검색
 → RRF는 순위만 결합, Re-Ranking은 리뷰 내용까지 이해

 검색어: '따뜻하고 가벼운 자켓'

📌 하이브리드 검색 (BM25 + Vector + RRF)
--------------------------------------------------------------------------------
1. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원 | RRF 점수: 0.0331
2. 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)...
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원 | RRF 점수: 0.0320
3. [공식/10년보증]마와 옷걸이 실루엣라이트 36FT 얇은셔츠용 10개...
   브랜드: 마와(백화점) | 카테고리: 생활/건강
   가격: 34,000원 | RRF 점수: 0.0318
4. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 65,550원 | RRF 점수: 0.0315
5. [헤지스핸드백] HJSS5E017 남성 화이트 배색 아가일 체크 장목 양말 3종 세트...
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 16,240원 | RRF 점수: 0.0314

📌 하이브리드 + Semantic Re-Ranking
--------------------------------------------------------------------------------
1. 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)...
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원 | Reranker 점수: 2.7018
   💬 Caption: 노스페이스 N

### 실습 2: 복합 의도 파악이 필요한 검색

**비즈니스 상황**: "아이 생일 선물용으로 좋은 제품" - 단순 키워드 매칭이 아닌 "선물용으로 적합한" 의미 파악 필요

- **하이브리드(RRF)**: "아이", "생일", "선물" 각각의 키워드+벡터 순위 결합
- **Re-Ranking**: "선물용으로 좋아요", "아이가 좋아해요" 같은 맥락 이해

In [18]:
print("\n" + "="*80)
print(" 시나리오 2: 복합 의도 파악")
print(" → RRF는 개별 단어/벡터 순위 결합, Re-Ranking은 전체 의도 이해")
print("="*80)

hybrid_2, semantic_2 = search_comparison("아이 생일 선물용으로 좋은 제품", top=5)

print("\n💡 분석:")
print("   - 하이브리드(RRF): '아이', '생일', '선물' 각각의 매칭 순위 결합")
print("   - Re-Ranking: '선물용으로 좋아요' 같은 실제 사용 후기 맥락 이해")
print("   → 선물 목적에 적합하다는 평가가 있는 상품이 상위로 이동")


 시나리오 2: 복합 의도 파악
 → RRF는 개별 단어/벡터 순위 결합, Re-Ranking은 전체 의도 이해

 검색어: '아이 생일 선물용으로 좋은 제품'

📌 하이브리드 검색 (BM25 + Vector + RRF)
--------------------------------------------------------------------------------
1. [멜리멜로] 생일/집들이선물 클래식 스톤 디퓨저 생일 집들이 친구선물...
   브랜드: 멜리멜로 | 카테고리: 인테리어
   가격: 28,000원 | RRF 점수: 0.0331
2. [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형...
   브랜드: 블루독베이비 | 카테고리: 유아동
   가격: 36,000원 | RRF 점수: 0.0318
3. 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   브랜드: 압소바 | 카테고리: 유아동
   가격: 39,000원 | RRF 점수: 0.0315
4. [파우치 증정] 선쿠션+클렌징패드 (조카선물 스킨케어선물)...
   브랜드: 노베즈 | 카테고리: 유아동
   가격: 54,000원 | RRF 점수: 0.0297
5. [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카...
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원 | RRF 점수: 0.0287

📌 하이브리드 + Semantic Re-Ranking
--------------------------------------------------------------------------------
1. 마이크로킥보드_스쿠터 스피드 디럭스 (색상선택)...
   브랜드: 마이크로킥보드 | 카테고리: 유아동
   가격: 275,000원 | Reranker 점수: 2.8557
   💬 Caption: 아이 4살 생일 선물로 마이크로킥보드 스피드 디럭스 구매했어요. 디자인이 세련되고

### 실습 3: 상품 특성 기반 검색

**비즈니스 상황**: "활동적인 운동할 때 입기 좋은 옷" - 운동 시 실제 사용감에 대한 이해 필요

- **하이브리드(RRF)**: "활동", "운동", "옷" 키워드+벡터 순위 결합
- **Re-Ranking**: "운동할 때 편해요", "신축성이 좋아요" 같은 실제 특성 이해

In [19]:
print("\n" + "="*80)
print(" 시나리오 3: 상품 특성 기반 검색")
print(" → RRF는 단어 매칭, Re-Ranking은 실제 사용 특성 이해")
print("="*80)

hybrid_3, semantic_3 = search_comparison("활동적인 운동할 때 입기 좋은 옷", top=5)

print("\n💡 분석:")
print("   - 하이브리드(RRF): '활동', '운동' 키워드+벡터 순위 결합")
print("   - Re-Ranking: '신축성이 좋아요', '운동할 때 편해요' 같은 실제 사용감 이해")
print("   → 운동 시 착용감이 좋다는 평가가 있는 상품이 상위로 이동")


 시나리오 3: 상품 특성 기반 검색
 → RRF는 단어 매칭, Re-Ranking은 실제 사용 특성 이해

 검색어: '활동적인 운동할 때 입기 좋은 옷'

📌 하이브리드 검색 (BM25 + Vector + RRF)
--------------------------------------------------------------------------------
1. [아디다스코리아공식] JC8149 4.0 3-스트라이프 루즈 크롭 티셔츠 3S LOOSE ...
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 24,500원 | RRF 점수: 0.0297
2. [나이키] DF 피트니스 티셔츠 NIKE DX0990-010...
   브랜드: 나이키 | 카테고리: 스포츠/레져
   가격: 23,480원 | RRF 점수: 0.0296
3. [나이키] NSW 우븐 여성 MR 쇼츠 NIKE FV7558-010...
   브랜드: 나이키 | 카테고리: 스포츠/레져
   가격: 29,500원 | RRF 점수: 0.0295
4. BAMKEL 밤켈 모던 32QT 쿨러하드쿨러...
   브랜드: 밤켈 | 카테고리: 스포츠/레져
   가격: 210,000원 | RRF 점수: 0.0286
5. 젝시믹스 다이나믹 컴포트 오버핏 숏슬리브 XMK2TS1102...
   브랜드: 젝시믹스 | 카테고리: 스포츠/레져
   가격: 18,100원 | RRF 점수: 0.0284

📌 하이브리드 + Semantic Re-Ranking
--------------------------------------------------------------------------------
1. [아디다스코리아공식] JC8149 4.0 3-스트라이프 루즈 크롭 티셔츠 3S LOOSE ...
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 24,500원 | Reranker 점수: 2.5764
   💬 Caption: 아디다스 3-스트라이프 루즈 크롭 티셔츠는 디자인이 정말 멋스럽고 편안

### 실습 4: 품질/만족도 기반 검색

**비즈니스 상황**: "품질 좋고 만족도 높은 브랜드 제품" - 실제 고객 만족도 평가 이해 필요

- **하이브리드(RRF)**: "품질", "만족도", "브랜드" 키워드+벡터 순위 결합
- **Re-Ranking**: "품질이 정말 좋아요", "만족스러워요" 같은 실제 평가 이해

In [20]:
print("\n" + "="*80)
print(" 시나리오 4: 품질/만족도 기반 검색")
print(" → RRF는 '품질' 단어 매칭, Re-Ranking은 실제 품질 평가 이해")
print("="*80)

hybrid_4, semantic_4 = search_comparison("품질 좋고 만족도 높은 제품", top=5)

print("\n💡 분석:")
print("   - 하이브리드(RRF): '품질', '만족도' 키워드+벡터 순위 결합")
print("   - Re-Ranking: '품질이 좋아요', '만족스러워요' 같은 실제 고객 평가 이해")
print("   → 긍정적인 품질 평가가 있는 상품이 상위로 이동")


 시나리오 4: 품질/만족도 기반 검색
 → RRF는 '품질' 단어 매칭, Re-Ranking은 실제 품질 평가 이해

 검색어: '품질 좋고 만족도 높은 제품'

📌 하이브리드 검색 (BM25 + Vector + RRF)
--------------------------------------------------------------------------------
1. [공식/10년보증]프리미엄 마와 옷걸이 굿페어 세트 (25개구성) - 화이트...
   브랜드: 마와(백화점) | 카테고리: 생활/건강
   가격: 109,000원 | RRF 점수: 0.0283
2. 설화수[8월]윤조에센스 6세대 90ml 기획세트...
   브랜드: 설화수 | 카테고리: 이미용
   가격: 140,000원 | RRF 점수: 0.0263
3. 설화수[8월]퍼펙팅 쿠션 에어리 본품15g+리필15g SPF50+...
   브랜드: 설화수 | 카테고리: 이미용
   가격: 89,000원 | RRF 점수: 0.0241
4. [클라랑스]맨 스킨&로션 세트...
   브랜드: 클라랑스 | 카테고리: 이미용
   가격: 118,000원 | RRF 점수: 0.0221
5. 딥티크 오드퍼퓸 오르페옹 75ml...
   브랜드: 딥티크 | 카테고리: 이미용
   가격: 289,000원 | RRF 점수: 0.0216

📌 하이브리드 + Semantic Re-Ranking
--------------------------------------------------------------------------------
1. 모던비스트 레드 홀리데이 장난감 세트 키티 MDB015RD...
   브랜드: 모던비스트 | 카테고리: 문화/취미
   가격: 70,000원 | Reranker 점수: 2.7217
   💬 Caption: 모던비스트 레드 홀리데이 장난감 세트 키티 MDB015RD를 구매했습니다. 모던비스트 제품답게 품질이 좋고 만족스럽습니다. 문화/취미 제품으로 추천합니다.. 모던비스트....
2

### 실습 5: 용도/목적 기반 검색

**비즈니스 상황**: "데일리로 매일 입기 좋은 편한 옷" - 일상 착용 적합성 평가 필요

- **하이브리드(RRF)**: "데일리", "매일", "편한" 키워드+벡터 순위 결합
- **Re-Ranking**: "매일 입어요", "편하게 잘 입고 있어요" 같은 실제 사용 패턴 이해

In [24]:
print("\n" + "="*80)
print(" 시나리오 5: 용도/목적 기반 검색")
print(" → RRF는 '데일리' 단어 매칭, Re-Ranking은 실제 일상 착용 후기 이해")
print("="*80)

hybrid_5, semantic_5 = search_comparison("데일리로 매일 입기 좋은 편한 옷", top=5)

print("\n💡 분석:")
print("   - 하이브리드(RRF): '데일리', '매일', '편한' 키워드+벡터 순위 결합")
print("   - Re-Ranking: '매일 입어요', '편하게 잘 입고 있어요' 같은 실제 사용 패턴 이해")
print("   → 일상 착용에 적합하다는 평가가 있는 상품이 상위로 이동")


 시나리오 5: 용도/목적 기반 검색
 → RRF는 '데일리' 단어 매칭, Re-Ranking은 실제 일상 착용 후기 이해

 검색어: '데일리로 매일 입기 좋은 편한 옷'

📌 하이브리드 검색 (BM25 + Vector + RRF)
--------------------------------------------------------------------------------
1. [로우백] 홈웨어 데일리 복서 쇼츠 (라일락 핑크) HOME WEAR BOXERS SHOR...
   브랜드: 로우백 | 카테고리: 언더웨어
   가격: 37,110원 | RRF 점수: 0.0318
2. [공식] 롱샴 핸들 파우치 34175089P68...
   브랜드: 롱샴 | 카테고리: 패션잡화
   가격: 160,000원 | RRF 점수: 0.0273
3. 라코스테우먼 25SS 여성 베이직 린넨셔츠 CF905E-55G AFS...
   브랜드: 라코스테 우먼 | 카테고리: 패션의류
   가격: 137,970원 | RRF 점수: 0.0273
4. 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)...
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 65,550원 | RRF 점수: 0.0264
5. 라코스테우먼 25SS 여성 스트라이프 긴팔 티셔츠 TF922E-55G HBP...
   브랜드: 라코스테 우먼 | 카테고리: 패션의류
   가격: 93,870원 | RRF 점수: 0.0259

📌 하이브리드 + Semantic Re-Ranking
--------------------------------------------------------------------------------
1. [로우백] 홈웨어 데일리 복서 쇼츠 (라일락 핑크) HOME WEAR BOXERS SHOR...
   브랜드: 로우백 | 카테고리: 언더웨어
   가격: 37,110원 | Reranker 점수: 2.5374
   💬 Caption: 로우백 홈웨어 데일

---
## 7️⃣ Semantic Answers 활용

질문형 쿼리에 대해 리뷰/상품 설명에서 직접 답변을 추출하는 기능입니다.

In [22]:
def search_with_answers(query, top=3):
    """하이브리드 + Semantic Answers 검색"""
    
    print(f"\n{'='*80}")
    print(f" 질문: '{query}'")
    print(f"{'='*80}")
    
    # 쿼리 임베딩 생성
    query_vector = get_embedding(query)
    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=50, fields="content_vector")
    
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="products-semantic-config",
        query_caption=QueryCaptionType.EXTRACTIVE,
        query_answer=QueryAnswerType.EXTRACTIVE,  # Answers 활성화
        top=top
    )
    
    # Answers 출력 (결과 전체에서 추출)
    answers = results.get_answers()
    if answers:
        print("\n💡 Semantic Answers (직접 답변):")
        print("-" * 80)
        for answer in answers:
            print(f"   ✅ {answer.text}")
            print(f"      (점수: {answer.score:.4f})")
    else:
        print("\n⚠️ 질문에 대한 직접 답변을 찾지 못했습니다.")
    
    # 검색 결과 출력
    print("\n📊 관련 상품:")
    print("-" * 80)
    for idx, result in enumerate(results, 1):
        print(f"{idx}. {result['title'][:50]}...")
        print(f"   브랜드: {result['brand']} | 카테고리: {result['category']}")
        print(f"   Reranker 점수: {result.get('@search.reranker_score', 0):.4f}")
        
        # Caption
        captions = result.get('@search.captions')
        if captions and len(captions) > 0:
            caption_text = captions[0].text
            if caption_text:
                print(f"   💬 {caption_text[:150]}...")
        print()

In [25]:
# 질문형 쿼리 예시
search_with_answers("피부를 위해서 어떤 브랜드가 인기있나요?")


 질문: '피부를 위해서 어떤 브랜드가 인기있나요?'

💡 Semantic Answers (직접 답변):
--------------------------------------------------------------------------------
   ✅ 설화수 에센셜 데일리 세트 자음2종을 8월에 구매해서 한 달간 사용해봤어요. 가볍지만 깊은 보습감이 느껴져서 여름철에도 끈적임 없이 산뜻하게 사용할 수 있었어요. 특히 자음수와 자음유액 조합이 피부를 촉촉하고 탄탄하게 잡아줘서 아침저녁으로 꾸준히 사용하니 피부결이 한층 부드러워진 게 느껴졌습니다. 고급스러운 향도 기분 전환에 좋고, 용기도 세련돼서 욕실 분위기가 한층 업그레이드됐어요. 가격대는 조금 있지만 그만큼 만족스러운 제품이라 추천합니다. 설화수.
      (점수: 0.8990)

📊 관련 상품:
--------------------------------------------------------------------------------
1. 샤넬 르 르쿠르브 실 드 샤넬...
   브랜드: 샤넬 | 카테고리: 이미용
   Reranker 점수: 2.0632
   💬 샤넬 르 르쿠르브 실 드 샤넬을 구매하고 나서 눈가와 입가 주름이 조금씩 완화되는 느낌을 받았어요. 가벼운 텍스처라 끈적임 없이 흡수가 빠르고, 바른 뒤 피부가 매끈해져서 메이크업 전에 사용하기도 좋아요. 가격대가 조금 있지만 고급스러운 향과 효과 덕분에 만족스럽습니다...

2. [디올] 포에버 스킨 글로우 24H 웨어 래디언트 파운데이션...
   브랜드: 크리스찬 디올 | 카테고리: 이미용
   Reranker 점수: 2.0183
   💬 디올 포에버 스킨 글로우 파운데이션은 처음 써보는데 피부에 정말 자연스럽게 밀착되네요. 24시간 지속력이라고 해서 반신반의했지만 하루 종일 번들거림 없이 촉촉함이 유지돼서 만족스러웠어요. 커버력도 적당해서 잡티를 가려주면서도 두꺼워 보이지 않고, 광도 은은하게 나서 피...

3. [클라랑스

In [ ]:
# 질문형 쿼리 예시 2
search_with_answers("유아용품 중에서 가장 좋은 제품은?")

---
## 8️⃣ 실무 시나리오: 카테고리 필터 + Re-Ranking

카테고리 필터와 Re-Ranking을 함께 사용하여 더 정확한 검색 결과를 제공합니다.

In [ ]:
print("\n" + "="*80)
print(" 실무 시나리오: 카테고리 필터 + Semantic 검색")
print("="*80)

# 시나리오 1: 스포츠/레져 카테고리에서 "활동적인 운동에 좋은 제품"
print("\n📍 시나리오 1: 스포츠/레져 - 활동적인 운동에 적합한 제품")
print("-" * 80)

results = search_client.search(
    search_text="활동적인 운동에 좋은 제품",
    filter="category eq '스포츠/레져'",
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name="products-semantic-config",
    query_caption=QueryCaptionType.EXTRACTIVE,
    top=5
)

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['title'][:45]}...")
    print(f"   브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
    print(f"   Reranker 점수: {result.get('@search.reranker_score', 0):.4f}")
    
    captions = result.get('@search.captions')
    if captions and len(captions) > 0:
        print(f"   💬 {captions[0].text[:120]}...")
    print()

In [ ]:
# 시나리오 2: 패션잡화 카테고리에서 "세련되고 고급스러운 디자인"
print("\n📍 시나리오 2: 패션잡화 - 세련되고 고급스러운 디자인")
print("-" * 80)

results = search_client.search(
    search_text="세련되고 고급스러운 디자인",
    filter="category eq '패션잡화'",
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name="products-semantic-config",
    query_caption=QueryCaptionType.EXTRACTIVE,
    top=5
)

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['title'][:45]}...")
    print(f"   브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
    print(f"   Reranker 점수: {result.get('@search.reranker_score', 0):.4f}")
    
    captions = result.get('@search.captions')
    if captions and len(captions) > 0:
        print(f"   💬 {captions[0].text[:120]}...")
    print()

---
## 9️⃣ 성능 비교: 하이브리드 vs 하이브리드 + Re-Ranking

In [ ]:
import time

def compare_performance(query, top=10):
    """하이브리드 vs 하이브리드+Re-Ranking 성능 비교"""
    
    print(f"\n{'='*80}")
    print(f" 성능 비교: '{query}'")
    print(f"{'='*80}")
    
    # 쿼리 임베딩 생성
    query_vector = get_embedding(query)
    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=50, fields="content_vector")
    
    # 하이브리드 검색 (RRF만)
    start = time.time()
    hybrid_results = list(search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        top=top
    ))
    hybrid_time = time.time() - start
    
    # 하이브리드 + Re-Ranking
    start = time.time()
    semantic_results = list(search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="products-semantic-config",
        top=top
    ))
    semantic_time = time.time() - start
    
    print("\n📊 성능 지표:")
    print("-" * 70)
    print(f"{'검색 모드':<25} {'응답 시간':<15} {'결과 개수':<15}")
    print("-" * 70)
    print(f"{'Hybrid (RRF만)':<25} {hybrid_time*1000:<14.2f}ms {len(hybrid_results):<15}")
    print(f"{'Hybrid + Re-Ranking':<25} {semantic_time*1000:<14.2f}ms {len(semantic_results):<15}")
    print("-" * 70)
    print(f"추가 시간: {(semantic_time - hybrid_time)*1000:+.2f}ms (Re-Ranking 처리)")
    
    # 순위 변화 분석
    print("\n🔄 상위 5개 결과 순위 변화:")
    print("-" * 90)
    print(f"{'순위':<5} {'Hybrid(RRF) 상품':<45} {'+ Re-Ranking 상품':<45}")
    print("-" * 90)
    
    for i in range(min(5, len(hybrid_results), len(semantic_results))):
        h_title = hybrid_results[i]['title'][:40] + "..."
        s_title = semantic_results[i]['title'][:40] + "..."
        print(f"{i+1:<5} {h_title:<45} {s_title:<45}")

# 성능 비교 실행
compare_performance("편안하고 따뜻한 겨울 옷", top=10)

---
## 🔟 정리 및 핵심 포인트

### ✅ 학습한 내용

1. **하이브리드 검색(RRF)의 한계**
   - RRF는 BM25와 Vector의 **순위만 결합**
   - 쿼리와 문서의 **실제 의미적 관련성**은 평가하지 못함
   - 복합적인 쿼리 의도 파악에 한계

2. **Semantic Re-Ranking이 해결하는 문제**
   - 하이브리드 검색 결과(최대 50개)를 언어 모델로 **재평가**
   - 쿼리와 문서 내용의 **실제 의미적 관련성** 기준으로 재정렬
   - 리뷰, 상품 설명 등의 **맥락 이해**

3. **비즈니스 시나리오**
   - 리뷰 기반 품질 검색: "따뜻하고 가벼운" → 실제 그런 평가가 있는 상품 우선
   - 복합 의도 파악: "선물용으로 좋은" → 선물 적합성 평가 이해
   - 사용 특성 검색: "운동할 때 편한" → 실제 사용감 리뷰 이해

4. **Captions & Answers**
   - Captions: 쿼리와 관련된 문장 자동 추출
   - Answers: 질문에 대한 직접 답변 추출

### ⚠️ 주의사항

- **비용**: Semantic Search는 추가 요금 발생 (Standard: $0.50/1000 쿼리)
- **제한**: 최대 50개 문서만 re-ranking 대상 (하이브리드 검색 상위 50개)
- **응답 시간**: 하이브리드 대비 +50~100ms 추가 (언어 모델 처리)

### 🎯 언제 Re-Ranking을 사용하나?

| 상황 | 하이브리드(RRF)만 | + Re-Ranking |
|------|------------------|--------------|
| 정확한 제품명 검색 | ✅ 충분 | 불필요 |
| 리뷰 기반 품질 검색 | 한계 있음 | ✅ 권장 |
| 복합 의도 쿼리 | 한계 있음 | ✅ 권장 |
| 단순 필터 검색 | ✅ 충분 | 불필요 |
| 비용 민감한 대량 검색 | ✅ 권장 | 선택적 |

### 💡 핵심 요약

```
하이브리드 검색(RRF) = 키워드 매칭 순위 + 벡터 유사도 순위 결합
                       ↓
            "순위"만 결합, 실제 의미는 모름
                       ↓
            Re-Ranking = 언어 모델로 의미적 관련성 재평가
                       ↓
            쿼리 의도에 더 정확히 부합하는 결과 상위 노출
```

In [ ]:
# (참고) Semantic Configuration 제거 방법
# existing_index = index_client.get_index(INDEX_NAME)
# existing_index.semantic_search = None
# index_client.create_or_update_index(existing_index)
# print("✅ Semantic Configuration 제거 완료")

print("💡 Semantic Configuration은 인덱스에 영구적으로 저장됩니다.")
print("💡 제거하려면 위의 주석 코드를 참고하세요.")
print("\n✅ Semantic Re-Ranking 실습 완료!")